## Masking

1) Causal Mask
1) Dropout Mask

In [15]:
import torch

print(torch.__version__)

2.13.0+cpu


In [16]:
inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89], # Your x1
        [0.55, 0.87, 0.66], # journey x2
        [0.57, 0.85, 0.64], # starts x3
        [0.22, 0.58, 0.33], # with x4
        [0.77, 0.25, 0.10], # one x5
        [0.05, 0.80, 0.55] # step x6
    ]
)

In [17]:
inputs.shape

torch.Size([6, 3])

In [18]:
d_in = inputs.shape[1] # the input embedding size, d = 3
d_out = 2

In [19]:
d_in

3

In [20]:
d_out

2

In [21]:
batch = torch.stack((inputs, inputs), dim = 0)

batch.shape

torch.Size([2, 6, 3])

In [22]:
import torch.nn as nn

class CausalAttention(nn.Module):

    def __init__(self, input_dimension, output_dimension, context_length, dropout, qkv_bias = False):

        super().__init__()

        self.W_query = torch.nn.Linear(input_dimension, output_dimension, bias = qkv_bias)
        self.W_key = torch.nn.Linear(input_dimension, output_dimension, bias = qkv_bias)
        self.W_value = torch.nn.Linear(input_dimension, output_dimension, bias = qkv_bias)

        self.dropout = torch.nn.Dropout(dropout) # dropout mask
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal = 1))

    def forward(self, x):

        b, num_tokens, input_dimension = x.shape
        # x = batch -> (2 x 6 x 3)
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        attention_scores = queries @ keys.transpose(1,2) #changed transpose
        attention_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        
        attention_weights = torch.softmax(attention_scores / keys.shape[-1]**0.5, dim = 1)
        attention_weights = self.dropout(attention_weights)

        context_vectors = attention_weights @ values

        return context_vectors

torch.manual_seed(789)

context_length = batch.shape[1]
dropout = 0.0

causal_attention = CausalAttention(d_in, d_out, context_length, dropout)

In [23]:
causal_attention(batch)

tensor([[[-0.0140,  0.0046],
         [-0.0392,  0.0209],
         [-0.0655,  0.0449],
         [-0.0915,  0.0415],
         [-0.0220,  0.2167],
         [-0.2306,  0.0700]],

        [[-0.0140,  0.0046],
         [-0.0392,  0.0209],
         [-0.0655,  0.0449],
         [-0.0915,  0.0415],
         [-0.0220,  0.2167],
         [-0.2306,  0.0700]]], grad_fn=<UnsafeViewBackward0>)